# IFBC Buddhist Texts Corpus Scraper

This notebook systematically downloads all Buddhist texts from the IFBC website (https://download.ifbcnet.org/) while maintaining comprehensive metadata for NLP research.

## Features
- ✅ Resumable downloads (won't re-download existing files)
- ✅ Comprehensive metadata extraction
- ✅ Real-time progress tracking with category and book names
- ✅ Google Drive integration
- ✅ Automatic page counting
- ✅ Author template CSV generation
- ✅ Error handling and logging

## Output Structure
```
data/
├── raw/{category}/book_name.pdf
└── metadata/
    ├── metadata.json
    ├── download_progress.json
    ├── failed_downloads.json
    └── author_template.csv
```

## 1. Setup and Installation

In [1]:
# Install required packages
!pip install -q beautifulsoup4 requests gdown PyPDF2 tqdm pandas lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 8.4 MB/s eta 0:00:00


In [2]:
# Import libraries
import os
import json
import re
import time
import hashlib
from datetime import datetime
from pathlib import Path
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup
import gdown
import PyPDF2
import pandas as pd
from tqdm.auto import tqdm
from google.colab import drive

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


## 2. Mount Google Drive

In [3]:
# Mount Google Drive
drive.mount('/content/drive')

# Configuration
PROJECT_DIR = '/content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/'

# Verify project directory exists
if os.path.exists(PROJECT_DIR):
    print(f"✓ Project directory found: {PROJECT_DIR}")
else:
    print(f"✗ Project directory not found: {PROJECT_DIR}")
    print("Please update PROJECT_DIR to match your Google Drive structure")

Mounted at /content/drive
✓ Project directory found: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/


## 3. Configuration and Helper Functions

In [4]:
# Configuration
BASE_URL = "https://download.ifbcnet.org/"
DATA_DIR = os.path.join(PROJECT_DIR, "data/raw")
METADATA_DIR = os.path.join(PROJECT_DIR, "data/metadata")

# Create directories
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(METADATA_DIR, exist_ok=True)

# File paths
METADATA_FILE = os.path.join(METADATA_DIR, "metadata.json")
PROGRESS_FILE = os.path.join(METADATA_DIR, "download_progress.json")
FAILED_FILE = os.path.join(METADATA_DIR, "failed_downloads.json")
AUTHOR_TEMPLATE = os.path.join(METADATA_DIR, "author_template.csv")

# Request settings
REQUEST_DELAY = 2  # Seconds between requests (be respectful)
TIMEOUT = 30  # Request timeout
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

print("✓ Configuration complete")
print(f"  Data directory: {DATA_DIR}")
print(f"  Metadata directory: {METADATA_DIR}")

✓ Configuration complete
  Data directory: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/raw
  Metadata directory: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/metadata


In [5]:
# Helper Functions

def load_json(filepath, default=None):
    """Load JSON file or return default if not exists"""
    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            return json.load(f)
    return default if default is not None else {}

def save_json(data, filepath):
    """Save data to JSON file"""
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def sanitize_filename(filename):
    """Remove invalid characters from filename"""
    # Remove or replace invalid characters
    filename = re.sub(r'[<>:"/\\|?*]', '_', filename)
    # Limit length
    if len(filename) > 200:
        filename = filename[:200]
    return filename.strip()

def calculate_sha256(filepath):
    """Calculate SHA-256 checksum of file"""
    sha256_hash = hashlib.sha256()
    with open(filepath, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()

def get_pdf_page_count(filepath):
    """Get number of pages in PDF"""
    try:
        with open(filepath, 'rb') as f:
            pdf_reader = PyPDF2.PdfReader(f)
            return len(pdf_reader.pages)
    except Exception as e:
        print(f"  Warning: Could not count pages - {e}")
        return None

def get_file_size_mb(filepath):
    """Get file size in MB"""
    return os.path.getsize(filepath) / (1024 * 1024)

def extract_google_drive_id(url):
    """Extract file ID from Google Drive URL"""
    patterns = [
        r'/file/d/([a-zA-Z0-9_-]+)',
        r'id=([a-zA-Z0-9_-]+)',
    ]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    return None

def parse_book_title(title_text):
    """Parse Sinhala and English book titles"""
    title_text = title_text.strip()

    # Try to find English text in parentheses
    match = re.search(r'(.+?)\s*\(\s*(.+?)\s*\)\s*$', title_text)

    if match:
        sinhala = match.group(1).strip()
        english = match.group(2).strip()
        return sinhala, english
    else:
        # If no pattern found, treat entire text as Sinhala
        return title_text, ""

def extract_date_from_url(url):
    """Extract publication date from URL pattern /YYYY/MM/DD/"""
    match = re.search(r'/(\d{4})/(\d{2})/(\d{2})/', url)
    if match:
        year, month, day = match.groups()
        return f"{year}-{month}-{day}"
    return None

print("✓ Helper functions loaded")

✓ Helper functions loaded


## 4. Web Scraping Functions

In [6]:
class IFBCScraper:
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update(HEADERS)
        self.metadata = load_json(METADATA_FILE, [])
        self.progress = load_json(PROGRESS_FILE, {'processed_urls': [], 'last_category': None})
        self.failed = load_json(FAILED_FILE, [])

    def get_page(self, url):
        """Fetch page with error handling"""
        try:
            time.sleep(REQUEST_DELAY)
            response = self.session.get(url, timeout=TIMEOUT)
            response.raise_for_status()
            return response.text
        except Exception as e:
            print(f"  Error fetching {url}: {e}")
            return None

    def discover_categories(self):
        """Discover all categories from the main page"""
        print("\n" + "="*60)
        print("DISCOVERING CATEGORIES")
        print("="*60)

        html = self.get_page(BASE_URL)
        if not html:
            return []

        soup = BeautifulSoup(html, 'html.parser')
        categories = []

        # Find all category links (adjust selector based on actual site structure)
        # Looking for links that contain '/category/'
        for link in soup.find_all('a', href=True):
            href = link['href']
            if '/category/' in href:
                full_url = urljoin(BASE_URL, href)
                if full_url not in categories:
                    categories.append(full_url)
                    category_name = link.get_text(strip=True)
                    print(f"  Found category: {category_name}")

        print(f"\n✓ Discovered {len(categories)} categories")
        return categories

    def get_posts_from_category(self, category_url):
        """Get all posts from a category (may include sub-categories or direct PDFs)"""
        html = self.get_page(category_url)
        if not html:
            return []

        soup = BeautifulSoup(html, 'html.parser')
        posts = []

        # Find all post links (adjust selector based on actual site structure)
        # Looking for article/post links
        for article in soup.find_all(['article', 'div'], class_=lambda x: x and ('post' in x.lower() or 'entry' in x.lower())):
            link = article.find('a', href=True)
            if link:
                post_url = urljoin(BASE_URL, link['href'])
                posts.append(post_url)

        # Also check for direct links in the page
        for link in soup.find_all('a', href=True):
            href = link['href']
            # Check if it's a post link (contains date pattern)
            if re.search(r'/\d{4}/\d{2}/\d{2}/', href):
                full_url = urljoin(BASE_URL, href)
                if full_url not in posts:
                    posts.append(full_url)

        return posts

    def extract_download_info(self, post_url):
        """Extract book information and download link from a post"""
        html = self.get_page(post_url)
        if not html:
            return None

        soup = BeautifulSoup(html, 'html.parser')

        # Extract title
        title_elem = soup.find(['h1', 'h2'], class_=lambda x: x and ('title' in x.lower() or 'entry' in x.lower()))
        if not title_elem:
            title_elem = soup.find(['h1', 'h2'])

        title = title_elem.get_text(strip=True) if title_elem else "Unknown Title"

        # Parse title into Sinhala and English
        book_name_si, book_name_en = parse_book_title(title)

        # Find Google Drive link
        drive_link = None
        for link in soup.find_all('a', href=True):
            href = link['href']
            if 'drive.google.com' in href:
                drive_link = href
                break

        if not drive_link:
            return None

        # Extract date from URL
        published_date = extract_date_from_url(post_url)

        return {
            'book_name_si': book_name_si,
            'book_name_en': book_name_en,
            'download_url': drive_link,
            'post_url': post_url,
            'published_date': published_date
        }

    def download_from_google_drive(self, drive_url, output_path):
        """Download file from Google Drive"""
        try:
            file_id = extract_google_drive_id(drive_url)
            if not file_id:
                print(f"  Error: Could not extract Google Drive file ID from {drive_url}")
                return False

            # Use gdown to download
            download_url = f"https://drive.google.com/uc?id={file_id}"
            gdown.download(download_url, output_path, quiet=False)

            # Verify file exists and has content
            if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
                return True
            else:
                return False

        except Exception as e:
            print(f"  Error downloading: {e}")
            return False

    def process_book(self, book_info, category_name):
        """Download and process a single book"""
        book_name_si = book_info['book_name_si']
        book_name_en = book_info['book_name_en']

        print(f"\n  Current Book: {book_name_si}")
        if book_name_en:
            print(f"               ({book_name_en})")

        # Create category directory
        category_dir = os.path.join(DATA_DIR, sanitize_filename(category_name))
        os.makedirs(category_dir, exist_ok=True)

        # Create filename
        if book_name_en:
            filename = sanitize_filename(f"{book_name_si}_{book_name_en}.pdf")
        else:
            filename = sanitize_filename(f"{book_name_si}.pdf")

        output_path = os.path.join(category_dir, filename)

        # Check if already downloaded
        if os.path.exists(output_path):
            print("  ✓ Already downloaded, skipping...")
            return None

        # Download
        print("  Downloading...")
        success = self.download_from_google_drive(book_info['download_url'], output_path)

        if not success:
            print("  ✗ Download failed")
            return None

        # Extract metadata
        print("  Extracting metadata...")
        page_count = get_pdf_page_count(output_path)
        file_size = get_file_size_mb(output_path)
        checksum = calculate_sha256(output_path)

        # Create metadata entry
        metadata_entry = {
            "BaseURL": BASE_URL,
            "Category": category_name,
            "Author": {
                "name": None,
                "DOB": None,
                "Date_of_passing": None
            },
            "book_name_si": book_name_si,
            "book_name_en": book_name_en,
            "number_of_pages": page_count,
            "published_date": book_info['published_date'],
            "download_url": book_info['download_url'],
            "post_url": book_info['post_url'],
            "local_path": output_path,
            "file_size_mb": round(file_size, 2),
            "sha256_checksum": checksum,
            "download_timestamp": datetime.now().isoformat()
        }

        print(f"  ✓ Downloaded successfully ({page_count} pages, {file_size:.2f} MB)")
        return metadata_entry

    def save_progress(self):
        """Save current progress"""
        save_json(self.metadata, METADATA_FILE)
        save_json(self.progress, PROGRESS_FILE)
        save_json(self.failed, FAILED_FILE)

    def run(self):
        """Main scraping process"""
        print("\n" + "="*60)
        print("IFBC BUDDHIST TEXTS SCRAPER")
        print("="*60)
        print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"Base URL: {BASE_URL}")
        print(f"Output directory: {DATA_DIR}")

        # Discover categories
        categories = self.discover_categories()

        if not categories:
            print("\n✗ No categories found. Please check the website structure.")
            return

        # Process each category
        for cat_idx, category_url in enumerate(categories, 1):
            category_name = category_url.split('/category/')[-1].rstrip('/')

            print("\n" + "="*60)
            print(f"PROCESSING CATEGORY {cat_idx}/{len(categories)}")
            print(f"Category: {category_name}")
            print(f"URL: {category_url}")
            print("="*60)

            # Skip if already processed
            if category_url in self.progress['processed_urls']:
                print("✓ Category already processed, skipping...")
                continue

            # Get posts from category
            posts = self.get_posts_from_category(category_url)
            print(f"\nFound {len(posts)} posts in this category")

            # Process each post
            for post_idx, post_url in enumerate(posts, 1):
                print(f"\n[{post_idx}/{len(posts)}] Processing post: {post_url}")

                # Skip if already processed
                if post_url in self.progress['processed_urls']:
                    print("  ✓ Already processed, skipping...")
                    continue

                try:
                    # Extract download info
                    book_info = self.extract_download_info(post_url)

                    if not book_info:
                        print("  ✗ No download link found")
                        self.failed.append({
                            'url': post_url,
                            'category': category_name,
                            'reason': 'No download link found',
                            'timestamp': datetime.now().isoformat()
                        })
                        continue

                    # Download and process book
                    metadata_entry = self.process_book(book_info, category_name)

                    if metadata_entry:
                        self.metadata.append(metadata_entry)

                    # Mark as processed
                    self.progress['processed_urls'].append(post_url)
                    self.progress['last_category'] = category_name

                    # Save progress periodically
                    if len(self.metadata) % 5 == 0:
                        self.save_progress()
                        print("  💾 Progress saved")

                except Exception as e:
                    print(f"  ✗ Error processing post: {e}")
                    self.failed.append({
                        'url': post_url,
                        'category': category_name,
                        'error': str(e),
                        'timestamp': datetime.now().isoformat()
                    })

            # Mark category as processed
            self.progress['processed_urls'].append(category_url)
            self.save_progress()

            print(f"\n✓ Category '{category_name}' completed")

        # Final save
        self.save_progress()

        # Summary
        print("\n" + "="*60)
        print("SCRAPING COMPLETE")
        print("="*60)
        print(f"Total books downloaded: {len(self.metadata)}")
        print(f"Failed downloads: {len(self.failed)}")
        print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"\nMetadata saved to: {METADATA_FILE}")
        print(f"Failed downloads saved to: {FAILED_FILE}")

print("✓ Scraper class loaded")

✓ Scraper class loaded


## 5. Generate Author Template CSV

In [7]:
def generate_author_template():
    """Generate CSV template for manual author data entry"""
    print("\nGenerating author template CSV...")

    # Load metadata
    metadata = load_json(METADATA_FILE, [])

    if not metadata:
        print("No metadata found. Run the scraper first.")
        return

    # Create DataFrame
    template_data = []
    for entry in metadata:
        template_data.append({
            'book_name_si': entry['book_name_si'],
            'book_name_en': entry['book_name_en'],
            'category': entry['Category'],
            'published_date': entry['published_date'],
            'author_name': '',
            'author_DOB': '',
            'author_date_of_passing': '',
            'notes': ''
        })

    df = pd.DataFrame(template_data)
    df.to_csv(AUTHOR_TEMPLATE, index=False, encoding='utf-8-sig')

    print(f"✓ Author template saved to: {AUTHOR_TEMPLATE}")
    print(f"  Total books: {len(template_data)}")
    print("\nInstructions:")
    print("1. Open the CSV file in Excel or Google Sheets")
    print("2. Fill in author information for each book")
    print("3. Use the merge function below to update metadata.json")

    return df

print("✓ Author template function loaded")

✓ Author template function loaded


In [8]:
def merge_author_data():
    """Merge completed author CSV data back into metadata.json"""
    print("\nMerging author data...")

    if not os.path.exists(AUTHOR_TEMPLATE):
        print(f"✗ Author template not found: {AUTHOR_TEMPLATE}")
        return

    # Load data
    metadata = load_json(METADATA_FILE, [])
    author_df = pd.read_csv(AUTHOR_TEMPLATE, encoding='utf-8-sig')

    # Create lookup dictionary
    author_lookup = {}
    for _, row in author_df.iterrows():
        key = (row['book_name_si'], row['book_name_en'])
        author_lookup[key] = {
            'name': row['author_name'] if pd.notna(row['author_name']) else None,
            'DOB': row['author_DOB'] if pd.notna(row['author_DOB']) else None,
            'Date_of_passing': row['author_date_of_passing'] if pd.notna(row['author_date_of_passing']) else None
        }

    # Update metadata
    updated_count = 0
    for entry in metadata:
        key = (entry['book_name_si'], entry['book_name_en'])
        if key in author_lookup:
            author_info = author_lookup[key]
            if author_info['name']:  # Only update if author name is provided
                entry['Author'] = author_info
                updated_count += 1

    # Save updated metadata
    save_json(metadata, METADATA_FILE)

    print(f"✓ Updated {updated_count} book entries with author information")
    print(f"  Metadata saved to: {METADATA_FILE}")

print("✓ Author merge function loaded")

✓ Author merge function loaded


## 6. Run the Scraper

In [ ]:
# Initialize and run scraper
scraper = IFBCScraper()
scraper.run()


IFBC BUDDHIST TEXTS SCRAPER
Start time: 2025-11-19 10:57:23
Base URL: https://download.ifbcnet.org/
Output directory: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/raw

DISCOVERING CATEGORIES
  Found category: 
  Found category: 
  Found category: 
  Found category: 
  Found category: 
  Found category: 
  Found category: 
  Found category: 
  Found category: 
  Found category: 
  Found category: 
  Found category: 
  Found category: 
  Found category: 
  Found category: 
  Found category: 

✓ Discovered 16 categories

PROCESSING CATEGORY 1/16
Category: thripitaka-books
URL: https://download.ifbcnet.org/category/thripitaka-books/
✓ Category already processed, skipping...

PROCESSING CATEGORY 2/16
Category: books-related-to-the-tipitaka
URL: https://download.ifbcnet.org/category/books-related-to-the-tipitaka/

Found 29 posts in this category

[1/29] Processing post: https://download.ifbcnet.org/2015/11/15/%e0%b6%b8%e0%b7%9b%e0%b6%ad%e0%b7%8a%e2%8

Downloading...
From: https://drive.google.com/uc?id=0B5RD8Fve5lhmVVFRY01XYmdEVjA
To: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/raw/books-related-to-the-tipitaka/මෛත්‍රී බුද්ධ වංශය නොහොත් අනාගත වංශය_Maithree Buddha Wanshaya nohoth anagatha wanshaya.pdf
100%|██████████| 3.94M/3.94M [00:00<00:00, 158MB/s]


  Extracting metadata...
  ✓ Downloaded successfully (35 pages, 3.76 MB)

[2/29] Processing post: https://download.ifbcnet.org/2015/11/15/%e0%b6%b8%e0%b7%9b%e0%b6%ad%e0%b7%8a%e2%80%8d%e0%b6%bb%e0%b7%93-%e0%b6%b6%e0%b7%94%e0%b6%af%e0%b7%8a%e0%b6%b0-%e0%b7%80%e0%b6%82%e0%b7%81%e0%b6%ba-%e0%b6%b1%e0%b7%9c%e0%b7%84%e0%b7%9c%e0%b6%ad/
  ✓ Already processed, skipping...

[3/29] Processing post: https://download.ifbcnet.org/2015/11/15/%e0%b6%b8%e0%b7%9b%e0%b6%ad%e0%b7%8a%e2%80%8d%e0%b6%bb%e0%b7%93-%e0%b6%b6%e0%b7%94%e0%b6%af%e0%b7%8a%e0%b6%b0-%e0%b7%80%e0%b6%82%e0%b7%81%e0%b6%ba-%e0%b6%b1%e0%b7%9c%e0%b7%84%e0%b7%9c%e0%b6%ad/
  ✓ Already processed, skipping...

[4/29] Processing post: https://download.ifbcnet.org/2015/11/15/%e0%b6%b8%e0%b7%9b%e0%b6%ad%e0%b7%8a%e2%80%8d%e0%b6%bb%e0%b7%93-%e0%b6%b6%e0%b7%94%e0%b6%af%e0%b7%8a%e0%b6%b0-%e0%b7%80%e0%b6%82%e0%b7%81%e0%b6%ba-%e0%b6%b1%e0%b7%9c%e0%b7%84%e0%b7%9c%e0%b6%ad/
  ✓ Already processed, skipping...

[5/29] Processing post: https://download.ifb

Downloading...
From: https://drive.google.com/uc?id=0B5RD8Fve5lhmejB4ZlhnekdvUGM
To: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/raw/books-related-to-the-tipitaka/පන්සිය පනස් ජාතක පොත් වහන්සේ_Pansiya panas Jathaka poth wahanse.pdf
100%|██████████| 25.0M/25.0M [00:00<00:00, 55.6MB/s]


  Extracting metadata...
  ✓ Downloaded successfully (998 pages, 23.80 MB)

[7/29] Processing post: https://download.ifbcnet.org/2015/11/15/%e0%b6%b4%e0%b6%b1%e0%b7%8a%e0%b7%83%e0%b7%92%e0%b6%ba-%e0%b6%b4%e0%b6%b1%e0%b7%83%e0%b7%8a-%e0%b6%a2%e0%b7%8f%e0%b6%ad%e0%b6%9a-%e0%b6%b4%e0%b7%9c%e0%b6%ad%e0%b7%8a-%e0%b7%80%e0%b7%84%e0%b6%b1/
  ✓ Already processed, skipping...

[8/29] Processing post: https://download.ifbcnet.org/2015/11/15/%e0%b6%b4%e0%b6%b1%e0%b7%8a%e0%b7%83%e0%b7%92%e0%b6%ba-%e0%b6%b4%e0%b6%b1%e0%b7%83%e0%b7%8a-%e0%b6%a2%e0%b7%8f%e0%b6%ad%e0%b6%9a-%e0%b6%b4%e0%b7%9c%e0%b6%ad%e0%b7%8a-%e0%b7%80%e0%b7%84%e0%b6%b1/
  ✓ Already processed, skipping...

[9/29] Processing post: https://download.ifbcnet.org/2015/03/04/%e0%b6%b8%e0%b7%84%e0%b7%8f-%e0%b7%83%e0%b6%ad%e0%b7%92%e0%b6%b4%e0%b6%a7%e0%b7%8a%e0%b6%a8%e0%b7%8f%e0%b6%b1-%e0%b7%83%e0%b7%96%e0%b6%ad%e0%b7%8a%e2%80%8d%e0%b6%bb-%e0%b7%80%e0%b6%bb%e0%b7%8a/

  Current Book: මහා සතිපට්ඨාන සූත්‍ර වර්ණනාව –
               (Maha Sathipa

...

  Extracting metadata...
  ✓ Downloaded successfully (None pages, 100.63 MB)

[7/14] Processing post: https://download.ifbcnet.org/2015/05/19/%e0%b6%85%e0%b6%b7%e0%b7%92%e0%b6%b0%e0%b6%bb%e0%b7%8a%e0%b6%b8-%e0%b6%b4%e0%b7%8a%e2%80%8d%e0%b6%bb%e0%b6%af%e0%b7%93%e0%b6%b4%e0%b7%92%e0%b6%9a%e0%b7%8f-%e0%b6%af%e0%b7%99%e0%b7%80%e0%b6%b1/
  ✓ Already processed, skipping...

[8/14] Processing post: https://download.ifbcnet.org/2015/05/19/%e0%b6%85%e0%b6%b7%e0%b7%92%e0%b6%b0%e0%b6%bb%e0%b7%8a%e0%b6%b8-%e0%b6%b4%e0%b7%8a%e2%80%8d%e0%b6%bb%e0%b6%af%e0%b7%93%e0%b6%b4%e0%b7%92%e0%b6%9a%e0%b7%8f-%e0%b6%af%e0%b7%99%e0%b7%80%e0%b6%b1/
  ✓ Already processed, skipping...

[9/14] Processing post: https://download.ifbcnet.org/2015/05/19/%e0%b6%85%e0%b6%b7%e0%b7%92%e0%b6%b0%e0%b6%bb%e0%b7%8a%e0%b6%b8-%e0%b6%b4%e0%b7%8a%e2%80%8d%e0%b6%bb%e0%b6%af%e0%b7%93%e0%b6%b4%e0%b7%92%e0%b6%9a%e0%b7%8f-%e0%b6%ad%e0%b7%94%e0%b6%b1%e0%b7%8a/

  Current Book: අභිධර්ම ප්‍රදීපිකා
               (තුන්වන පොත ) – අමරදාස රතනපා

Downloading...
From: https://drive.google.com/uc?id=0B5RD8Fve5lhmbUhOS2xTaGQ4SlU
To: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/raw/mr-amaradasa-rathanapala/අභිධර්ම ප්‍රදීපිකා_තුන්වන පොත ) – අමරදාස රතනපාල මහතා ( Abhidharma Pradeepika ( 3rd book ) – Mr. Amaradasa Rathanapala.pdf
100%|██████████| 94.4M/94.4M [00:01<00:00, 87.6MB/s]


  Extracting metadata...
  ✓ Downloaded successfully (None pages, 90.01 MB)

[10/14] Processing post: https://download.ifbcnet.org/2015/05/19/%e0%b6%85%e0%b6%b7%e0%b7%92%e0%b6%b0%e0%b6%bb%e0%b7%8a%e0%b6%b8-%e0%b6%b4%e0%b7%8a%e2%80%8d%e0%b6%bb%e0%b6%af%e0%b7%93%e0%b6%b4%e0%b7%92%e0%b6%9a%e0%b7%8f-%e0%b6%ad%e0%b7%94%e0%b6%b1%e0%b7%8a/
  ✓ Already processed, skipping...

[11/14] Processing post: https://download.ifbcnet.org/2015/05/19/%e0%b6%85%e0%b6%b7%e0%b7%92%e0%b6%b0%e0%b6%bb%e0%b7%8a%e0%b6%b8-%e0%b6%b4%e0%b7%8a%e2%80%8d%e0%b6%bb%e0%b6%af%e0%b7%93%e0%b6%b4%e0%b7%92%e0%b6%9a%e0%b7%8f-%e0%b6%ad%e0%b7%94%e0%b6%b1%e0%b7%8a/
  ✓ Already processed, skipping...

[12/14] Processing post: https://download.ifbcnet.org/2015/05/19/%e0%b6%85%e0%b6%b7%e0%b7%92%e0%b6%b0%e0%b6%bb%e0%b7%8a%e0%b6%b8-%e0%b6%b4%e0%b7%8a%e2%80%8d%e0%b6%bb%e0%b6%af%e0%b7%93%e0%b6%b4%e0%b7%92%e0%b6%9a%e0%b7%8f-%e0%b7%84%e0%b6%ad%e0%b6%bb%e0%b7%80/

  Current Book: අභිධර්ම ප්‍රදීපිකා
               (හතරවන පොත ) – අමරදාස රතනප

Downloading...
From (original): https://drive.google.com/uc?id=0B5RD8Fve5lhmZUNlSzdheU0tVkE
From (redirected): https://drive.google.com/uc?id=0B5RD8Fve5lhmZUNlSzdheU0tVkE&confirm=t&uuid=e083ce0c-9d5d-459e-8bb5-abb7c6d3615d
To: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/raw/mr-amaradasa-rathanapala/අභිධර්ම ප්‍රදීපිකා_හතරවන පොත ) – අමරදාස රතනපාල මහතා ( Abhidharma Pradeepika ( 4th book ) – Mr. Amaradasa Rathanapala.pdf
100%|██████████| 119M/119M [00:01<00:00, 78.9MB/s]


  Extracting metadata...
  ✓ Downloaded successfully (None pages, 113.22 MB)

[13/14] Processing post: https://download.ifbcnet.org/2015/05/19/%e0%b6%85%e0%b6%b7%e0%b7%92%e0%b6%b0%e0%b6%bb%e0%b7%8a%e0%b6%b8-%e0%b6%b4%e0%b7%8a%e2%80%8d%e0%b6%bb%e0%b6%af%e0%b7%93%e0%b6%b4%e0%b7%92%e0%b6%9a%e0%b7%8f-%e0%b7%84%e0%b6%ad%e0%b6%bb%e0%b7%80/
  ✓ Already processed, skipping...

[14/14] Processing post: https://download.ifbcnet.org/2015/05/19/%e0%b6%85%e0%b6%b7%e0%b7%92%e0%b6%b0%e0%b6%bb%e0%b7%8a%e0%b6%b8-%e0%b6%b4%e0%b7%8a%e2%80%8d%e0%b6%bb%e0%b6%af%e0%b7%93%e0%b6%b4%e0%b7%92%e0%b6%9a%e0%b7%8f-%e0%b7%84%e0%b6%ad%e0%b6%bb%e0%b7%80/
  ✓ Already processed, skipping...

✓ Category 'mr-amaradasa-rathanapala' completed

SCRAPING COMPLETE
Total books downloaded: 83
Failed downloads: 11
End time: 2025-11-19 11:13:29

Metadata saved to: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/metadata/metadata.json
Failed downloads saved to: /content/drive/Othercomputers/My 

## 7. Generate Author Template (Run after scraping)

In [ ]:
# Generate author template CSV
author_template_df = generate_author_template()

# Display first few rows
if author_template_df is not None:
    print("\nPreview of author template:")
    display(author_template_df.head())


Generating author template CSV...
✓ Author template saved to: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/metadata/author_template.csv
  Total books: 83

Instructions:
1. Open the CSV file in Excel or Google Sheets
2. Fill in author information for each book
3. Use the merge function below to update metadata.json

Preview of author template:


,book_name_si,book_name_en,category,published_date,author_name,author_DOB,author_date_of_passing,notes
0,ත්‍රිපිටකය – දීඝනිකාය,thripitaka – Digha Nikaya,thripitaka-books,2015-02-10,,,,
1,ත්‍රිපිටකය – මජ්ජිම නිකාය,thripitaka – Majjhima Nikay,thripitaka-books,2015-02-10,,,,
2,ත්‍රිපිටකය – සංයුක්ත නිකාය,thripitaka – Samyutta Nikaya,thripitaka-books,2015-02-10,,,,
3,ත්‍රිපිටකය – අංගුත්තර නිකාය,thripitaka – Anguttara Nikaya,thripitaka-books,2015-02-10,,,,
4,ත්‍රිපිටකය – ඛුද්ධක නිකාය,thripitaka – Khuddaka Nikaya,thripitaka-books,2015-02-10,,,,


## 8. View Statistics

In [ ]:
# Load and display statistics
metadata = load_json(METADATA_FILE, [])
failed = load_json(FAILED_FILE, [])

print("\n" + "="*60)
print("DOWNLOAD STATISTICS")
print("="*60)
print(f"Total books downloaded: {len(metadata)}")
print(f"Failed downloads: {len(failed)}")

if metadata:
    # Category breakdown
    categories = {}
    total_pages = 0
    total_size = 0

    for entry in metadata:
        cat = entry['Category']
        categories[cat] = categories.get(cat, 0) + 1
        if entry['number_of_pages']:
            total_pages += entry['number_of_pages']
        total_size += entry['file_size_mb']

    print(f"\nTotal pages: {total_pages:,}")
    print(f"Total size: {total_size:.2f} MB ({total_size/1024:.2f} GB)")
    print(f"\nBooks per category:")
    for cat, count in sorted(categories.items(), key=lambda x: x[1], reverse=True):
        print(f"  {cat}: {count}")

if failed:
    print(f"\nFailed downloads:")
    for item in failed[:10]:  # Show first 10
        print(f"  {item.get('url', 'Unknown')}: {item.get('reason', item.get('error', 'Unknown error'))}")


DOWNLOAD STATISTICS
Total books downloaded: 83
Failed downloads: 11

Total pages: 4,510
Total size: 3851.69 MB (3.76 GB)

Books per category:
  thripitaka-books: 10
  old-books: 10
  kukulpane-sudassi-himi: 10
  books-related-to-the-tipitaka: 9
  ven-rrajagiriye-ariyagnaana-thero-books: 9
  ven-kiribathgoda-gnanananda-thero-books: 7
  ven-rerukane-chandawimala-thero-books: 4
  ven-nauyane-ariyadhamma-maha-thero: 4
  ven-matara-sri-gnanarama-thero-books: 4
  buddhist-characters: 4
  mr-amaradasa-rathanapala: 4
  ven-katukurunde-gnanananda-thero-books: 3
  ven-galigamuwe-gnanadeepa-thero-books: 2
  ven-balangoda-ananda-maithriya-thero-books: 1
  ven-madihe-pannaseeha-thero-books: 1
  ven-nawalapitiye-ariyawansa-thero: 1

Failed downloads:
  https://download.ifbcnet.org/2015/11/13/%e0%b6%b6%e0%b7%9c%e0%b6%af%e0%b7%94-%e0%b6%9c%e0%b7%92%e0%b7%84%e0%b7%92-%e0%b6%b4%e0%b7%92%e0%b7%85%e0%b7%92%e0%b7%80%e0%b7%99%e0%b6%ad-%e0%b6%b6%e0%b7%85%e0%b7%8a%e0%b6%9c%e0%b7%9c/: No download link found
 

## 9. Retry Failed Downloads

In [ ]:
def retry_failed_downloads():
    """Retry downloading failed books"""
    failed = load_json(FAILED_FILE, [])

    if not failed:
        print("No failed downloads to retry.")
        return

    print(f"\nRetrying {len(failed)} failed downloads...\n")

    scraper = IFBCScraper()
    retry_failed = []

    for item in tqdm(failed, desc="Retrying"):
        url = item.get('url')
        if not url:
            continue

        try:
            # Remove from processed URLs to retry
            if url in scraper.progress['processed_urls']:
                scraper.progress['processed_urls'].remove(url)

            # Extract and process
            book_info = scraper.extract_download_info(url)
            if book_info:
                metadata_entry = scraper.process_book(book_info, item.get('category', 'Unknown'))
                if metadata_entry:
                    scraper.metadata.append(metadata_entry)
                    print(f"✓ Successfully retried: {book_info['book_name_si']}")
                else:
                    retry_failed.append(item)
            else:
                retry_failed.append(item)

        except Exception as e:
            print(f"✗ Retry failed for {url}: {e}")
            retry_failed.append(item)

    # Update failed list
    scraper.failed = retry_failed
    scraper.save_progress()

    print(f"\n✓ Retry complete")
    print(f"  Successfully downloaded: {len(failed) - len(retry_failed)}")
    print(f"  Still failed: {len(retry_failed)}")

# Uncomment to run retry
# retry_failed_downloads()

## 10. Merge Author Data (After filling the CSV)

In [9]:
# After you've filled in the author_template.csv, run this to merge the data
merge_author_data()


Merging author data...
✓ Updated 73 book entries with author information
  Metadata saved to: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/metadata/metadata.json
